In [2]:
!bash ./scripts/anomaly_detection/MSL/Autoformer.sh

Using GPU
Args in experiment:
Basic Config
  Task Name:          anomaly_detection   Is Training:        1                   
  Model ID:           MSL                 Model:              Autoformer          

Data Loader
  Data:               MSL                 Root Path:          ./dataset/MSL       
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Anomaly Detection Task
  Anomaly Ratio:      1.0                 

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             55                  Dec In:             7                   
  C Out:              55                  d model:            128                 
  n heads:            8                   e layers:           3                   
  d layers:           1                   d FF:               128    

In [1]:
# # !export CUDA_VISIBLE_DEVICES=0
# import os
# from tqdm import tqdm

# # Define the path
# for m in ['PSM', 'SMAP', 'SMD', 'SWAT']:
#     script_dir = f"scripts/anomaly_detection/{m}/"

#     # Get all .sh files and sort them
#     scripts = sorted([f for f in os.listdir(script_dir) if f.endswith('.sh')])

#     for script in tqdm(scripts, desc="Running anomaly detection scripts"):
#         print(f"--- Running {script} ---")
#         script_path = os.path.join(script_dir, script)
#         !bash {script_path}

In [13]:
import pandas as pd
df = pd.read_csv('dataset/LEAD_184_filtered.csv')
df.describe()

,building_id,meter_reading,anomaly
count,1.616256e+06,1.518236e+06,1.616256e+06
mean,8.072935e+02,1.877899e+02,2.231082e-02
std,4.073220e+02,4.098855e+02,1.476925e-01
min,4.100000e+01,0.000000e+00,0.000000e+00
25%,4.787500e+02,3.100000e+01,0.000000e+00
50%,9.070000e+02,8.700000e+01,0.000000e+00
75%,1.225250e+03,2.090000e+02,0.000000e+00
max,1.353000e+03,6.596890e+03,1.000000e+00


In [12]:
ratio = df.groupby("building_id")["anomaly"].mean()

print("Min anomaly ratio:", ratio.min())
print("Max anomaly ratio:", ratio.max())

Min anomaly ratio: 0.010132058287795993
Max anomaly ratio: 0.08822859744990892


In [ ]:
import pandas as pd
import os

df = pd.read_csv('dataset/lead_406_buildings_cleaned.csv')
remove_bid = [32, 534, 558, 653, 693, 723, 739, 855, 910, 970, 1147, 1183, 1264, 1282]
df = df[~df['building_id'].isin(remove_bid)]

df1 = pd.read_csv('dataset/LEAD_train_features.csv')
valid_buildings = df1['building_id'].unique()
df = df[df['building_id'].isin(valid_buildings)]

# Ensure temp directory exists
os.makedirs('temp', exist_ok=True)

for b_id in df['building_id'].unique()[:1]:
    print(f"--- Processing building_id: {b_id} ---")
    
    # Filter data for this building
    df_b = df[df['building_id'] == b_id].copy()

    actual_ratio = df_b['anomaly'].mean() * 100 
    
    print(f"Calculated Anomaly Ratio for Building {b_id}: {actual_ratio:.2f}%")
    
    # 2. Impute and Save
    df_b['meter_reading'] = df_b['meter_reading'].fillna(df_b['meter_reading'].median())
    df_b.to_csv('temp/train.csv', index=False)
    
    # 3. Pass the ratio as an argument to the bash script
    !python -u run.py \
        --task_name anomaly_detection \
        --is_training 1 \
        --root_path ./temp \
        --data_path train.csv \
        --model_id LEAD \
        --model Nonstationary_Transformer \
        --data LEAD \
        --features S \
        --seq_len 168 \
        --pred_len 0 \
        --d_model 64 \
        --d_ff 64 \
        --e_layers 2 \
        --enc_in 1 \
        --c_out 1 \
        # --anomaly_ratio {actual_ratio} \
        --batch_size 64 \
        --train_epochs 20 \
        --building_id {b_id} \
        --learning_rate 0.0001 \
        --patience 5

    print(f"Finished building_id: {b_id}\n")

--- Processing building_id: 1 ---
Calculated Anomaly Ratio for Building 1: 0.63%
Using GPU
Args in experiment:
Anomaly Detection Task

Traceback (most recent call last):
  File "/media/user/DATA21/Kajeeth/covariate_anomaly detection/time_series_library/run.py", line 193, in <module>
    from exp.exp_anomaly_detection_hyp import Exp_Anomaly_Detection
  File "/media/user/DATA21/Kajeeth/covariate_anomaly detection/time_series_library/exp/exp_anomaly_detection_hyp.py", line 1, in <module>
    from data_provider.data_factory import data_provider
  File "/media/user/DATA21/Kajeeth/covariate_anomaly detection/time_series_library/data_provider/data_factory.py", line 1, in <module>
    from data_provider.data_loader import LEADloader
  File "/media/user/DATA21/Kajeeth/covariate_anomaly detection/time_series_library/data_provider/data_loader.py", line 10, in <module>
    from data_provider.m4 import M4Dataset, M4Meta
  File "/media/user/DATA21/Kajeeth/covariate_anomaly detection/time_series_libr

In [16]:
import pandas as pd
import os

df = pd.read_csv('dataset/LEAD_191_filtered.csv')

# Ensure temp directory exists
os.makedirs('temp', exist_ok=True)

# 1. Filter the dataframe to only include the target buildings
target_buildings = [439, 683, 685, 687, 697, 884, 892]
df_filtered = df[df['building_id'].isin(target_buildings)]

# 2. Loop through the unique building IDs found in the filtered data
for b_id in df_filtered['building_id'].unique():
    print(f"--- Processing building_id: {b_id} ---")
    
    # Filter data for this specific building
    df_b = df_filtered[df_filtered['building_id'] == b_id].copy()

    # Calculate anomaly ratio
    actual_ratio = df_b['anomaly'].mean() * 100 
    
    print(f"Calculated Anomaly Ratio for Building {b_id}: {actual_ratio:.2f}%")
    
    # Impute and Save
    df_b['meter_reading'] = df_b['meter_reading'].fillna(df_b['meter_reading'].median())
    df_b.to_csv('temp/train.csv', index=False)
    
    # Pass the ratio as an argument to the bash script
    !python -u run.py \
        --task_name anomaly_detection \
        --is_training 1 \
        --root_path ./temp \
        --data_path train.csv \
        --model_id LEAD \
        --model MSGNet \
        --data LEAD \
        --features S \
        --seq_len 168 \
        --pred_len 0 \
        --d_model 64 \
        --d_ff 64 \
        --e_layers 2 \
        --enc_in 1 \
        --c_out 1 \
        --anomaly_ratio {actual_ratio} \
        --batch_size 64 \
        --train_epochs 20 \
        --building_id {b_id} \
        --learning_rate 0.0001 \
        --patience 5

    print(f"Finished building_id: {b_id}\n")

--- Processing building_id: 439 ---
Calculated Anomaly Ratio for Building 439: 6.45%
Using GPU
Args in experiment:
Anomaly Detection Task
  Anomaly Ratio:      6.454918032786885   

Use GPU: cuda:0
🚀 Lazy Loading: MSGNet ...
>>>>>>>start training : anomaly_detection_LEAD_MSGNet_LEAD_ftS_sl168_ll48_pl0_dm64_nh8_el2_dl1_df64_expand2_dc4_fc1_ebtimeF_dtTrue_test_0>>>>>>>>>>>>>>>>>>>>>>>>>>
train 4225
val 1297
test 2761
Epoch: 1 cost time: 4.8119401931762695
Epoch: 1, Steps: 67 | Train Loss: 0.0224046 Vali Loss: 0.0083183 Test Loss: 0.0077046
Validation loss decreased (inf --> 0.008318).  Saving model ...
Updating learning rate to 0.0001
Epoch: 2 cost time: 4.300130128860474
Epoch: 2, Steps: 67 | Train Loss: 0.0053542 Vali Loss: 0.0049505 Test Loss: 0.0034846
Validation loss decreased (0.008318 --> 0.004951).  Saving model ...
Updating learning rate to 5e-05
Epoch: 3 cost time: 4.547993421554565
Epoch: 3, Steps: 67 | Train Loss: 0.0033928 Vali Loss: 0.0039375 Test Loss: 0.0026667
Validation

In [17]:
import pandas as pd
import os

df = pd.read_csv('dataset/LEAD_191_filtered.csv')

# Ensure temp directory exists
os.makedirs('temp', exist_ok=True)

# 1. Filter the dataframe to only include the target buildings
target_buildings = [107, 173, 439, 926, 1257, 1284, 1319]
df_filtered = df[df['building_id'].isin(target_buildings)]

# 2. Loop through the unique building IDs found in the filtered data
for b_id in df_filtered['building_id'].unique():
    print(f"--- Processing building_id: {b_id} ---")
    
    # Filter data for this specific building
    df_b = df_filtered[df_filtered['building_id'] == b_id].copy()

    # Calculate anomaly ratio
    actual_ratio = df_b['anomaly'].mean() * 100 
    
    print(f"Calculated Anomaly Ratio for Building {b_id}: {actual_ratio:.2f}%")
    
    # Impute and Save
    df_b['meter_reading'] = df_b['meter_reading'].fillna(df_b['meter_reading'].median())
    df_b.to_csv('temp/train.csv', index=False)
    
    # Pass the ratio as an argument to the bash script
    !python -u run.py \
        --task_name anomaly_detection \
        --is_training 1 \
        --root_path ./temp \
        --data_path train.csv \
        --model_id LEAD \
        --model Informer \
        --data LEAD \
        --features S \
        --seq_len 168 \
        --pred_len 0 \
        --d_model 64 \
        --d_ff 64 \
        --e_layers 2 \
        --enc_in 1 \
        --c_out 1 \
        --anomaly_ratio {actual_ratio} \
        --batch_size 64 \
        --train_epochs 20 \
        --building_id {b_id} \
        --learning_rate 0.0001 \
        --patience 5

    print(f"Finished building_id: {b_id}\n")

--- Processing building_id: 107 ---
Calculated Anomaly Ratio for Building 107: 2.83%
Using GPU
Args in experiment:
Anomaly Detection Task
  Anomaly Ratio:      2.8346994535519126  

Use GPU: cuda:0
🚀 Lazy Loading: Informer ...
>>>>>>>start training : anomaly_detection_LEAD_Informer_LEAD_ftS_sl168_ll48_pl0_dm64_nh8_el2_dl1_df64_expand2_dc4_fc1_ebtimeF_dtTrue_test_0>>>>>>>>>>>>>>>>>>>>>>>>>>
train 4225
val 1297
test 2761
Epoch: 1 cost time: 0.8004424571990967
Epoch: 1, Steps: 67 | Train Loss: 0.2410750 Vali Loss: 0.0481439 Test Loss: 0.0493736
Validation loss decreased (inf --> 0.048144).  Saving model ...
Updating learning rate to 0.0001
Epoch: 2 cost time: 0.46251702308654785
Epoch: 2, Steps: 67 | Train Loss: 0.0519388 Vali Loss: 0.0327930 Test Loss: 0.0353934
Validation loss decreased (0.048144 --> 0.032793).  Saving model ...
Updating learning rate to 5e-05
Epoch: 3 cost time: 0.507781982421875
Epoch: 3, Steps: 67 | Train Loss: 0.0420257 Vali Loss: 0.0269392 Test Loss: 0.0299841
Vali

In [28]:
import pandas as pd
import os

df = pd.read_csv('dataset/LEAD_191_filtered.csv')

# Ensure temp directory exists
os.makedirs('temp', exist_ok=True)

# 1. Filter the dataframe to only include the target buildings
target_buildings = [1246, 1275]
df_filtered = df[df['building_id'].isin(target_buildings)]

# 2. Loop through the unique building IDs found in the filtered data
for b_id in df_filtered['building_id'].unique():
    print(f"--- Processing building_id: {b_id} ---")
    
    # Filter data for this specific building
    df_b = df_filtered[df_filtered['building_id'] == b_id].copy()

    # Calculate anomaly ratio
    actual_ratio = df_b['anomaly'].mean() * 100 
    
    print(f"Calculated Anomaly Ratio for Building {b_id}: {actual_ratio:.2f}%")
    
    # Impute and Save
    df_b['meter_reading'] = df_b['meter_reading'].fillna(df_b['meter_reading'].median())
    df_b.to_csv('temp/train.csv', index=False)
    
    # Pass the ratio as an argument to the bash script
    !python -u run.py \
        --task_name anomaly_detection \
        --is_training 1 \
        --root_path ./temp \
        --data_path train.csv \
        --model_id LEAD \
        --model KACformer \
        --data LEAD \
        --features S \
        --seq_len 168 \
        --pred_len 0 \
        --d_model 64 \
        --d_ff 64 \
        --e_layers 2 \
        --enc_in 1 \
        --c_out 1 \
        --anomaly_ratio {actual_ratio} \
        --batch_size 64 \
        --train_epochs 20 \
        --building_id {b_id} \
        --learning_rate 0.0001 \
        --patience 5

    print(f"Finished building_id: {b_id}\n")

--- Processing building_id: 1246 ---
Calculated Anomaly Ratio for Building 1246: 2.80%
Using GPU
Args in experiment:
Anomaly Detection Task
  Anomaly Ratio:      2.800546448087432   

Use GPU: cuda:0
🚀 Lazy Loading: KACformer ...
>>>>>>>start training : anomaly_detection_LEAD_KACformer_LEAD_ftS_sl168_ll48_pl0_dm64_nh8_el2_dl1_df64_expand2_dc4_fc1_ebtimeF_dtTrue_test_0>>>>>>>>>>>>>>>>>>>>>>>>>>
train 4225
val 1297
test 2761
Epoch: 1 cost time: 1.1992061138153076
Epoch: 1, Steps: 67 | Train Loss: 0.0048891 Vali Loss: 0.0039249 Test Loss: 0.0025936
Validation loss decreased (inf --> 0.003925).  Saving model ...
Updating learning rate to 0.0001
Epoch: 2 cost time: 0.8017857074737549
Epoch: 2, Steps: 67 | Train Loss: 0.0014438 Vali Loss: 0.0022840 Test Loss: 0.0015064
Validation loss decreased (0.003925 --> 0.002284).  Saving model ...
Updating learning rate to 5e-05
Epoch: 3 cost time: 0.8047633171081543
Epoch: 3, Steps: 67 | Train Loss: 0.0009979 Vali Loss: 0.0018837 Test Loss: 0.0012303


In [27]:
import pandas as pd
import os

df = pd.read_csv('dataset/LEAD_191_filtered.csv')

# Ensure temp directory exists
os.makedirs('temp', exist_ok=True)

# 1. Filter the dataframe to only include the target buildings
target_buildings = [1306]
df_filtered = df[df['building_id'].isin(target_buildings)]

# 2. Loop through the unique building IDs found in the filtered data
for b_id in df_filtered['building_id'].unique():
    print(f"--- Processing building_id: {b_id} ---")
    
    # Filter data for this specific building
    df_b = df_filtered[df_filtered['building_id'] == b_id].copy()

    # Calculate anomaly ratio
    actual_ratio = df_b['anomaly'].mean() * 100 
    
    print(f"Calculated Anomaly Ratio for Building {b_id}: {actual_ratio:.2f}%")
    
    # Impute and Save
    df_b['meter_reading'] = df_b['meter_reading'].fillna(df_b['meter_reading'].median())
    df_b.to_csv('temp/train.csv', index=False)
    
    # Pass the ratio as an argument to the bash script
    !python -u run.py \
        --task_name anomaly_detection \
        --is_training 1 \
        --root_path ./temp \
        --data_path train.csv \
        --model_id LEAD \
        --model KANformer \
        --data LEAD \
        --features S \
        --seq_len 168 \
        --pred_len 0 \
        --d_model 64 \
        --d_ff 64 \
        --e_layers 2 \
        --enc_in 1 \
        --c_out 1 \
        --anomaly_ratio {actual_ratio} \
        --batch_size 64 \
        --train_epochs 20 \
        --building_id {b_id} \
        --learning_rate 0.0001 \
        --patience 5

    print(f"Finished building_id: {b_id}\n")

--- Processing building_id: 1306 ---
Calculated Anomaly Ratio for Building 1306: 3.14%
Using GPU
Args in experiment:
Anomaly Detection Task
  Anomaly Ratio:      3.1420765027322406  

Use GPU: cuda:0
🚀 Lazy Loading: KANformer ...
>>>>>>>start training : anomaly_detection_LEAD_KANformer_LEAD_ftS_sl168_ll48_pl0_dm64_nh8_el2_dl1_df64_expand2_dc4_fc1_ebtimeF_dtTrue_test_0>>>>>>>>>>>>>>>>>>>>>>>>>>
train 4225
val 1297
test 2761
Epoch: 1 cost time: 1.230247974395752
Epoch: 1, Steps: 67 | Train Loss: 0.0016875 Vali Loss: 0.0005616 Test Loss: 0.0005613
Validation loss decreased (inf --> 0.000562).  Saving model ...
Updating learning rate to 0.0001
Epoch: 2 cost time: 0.802893877029419
Epoch: 2, Steps: 67 | Train Loss: 0.0002701 Vali Loss: 0.0002213 Test Loss: 0.0002125
Validation loss decreased (0.000562 --> 0.000221).  Saving model ...
Updating learning rate to 5e-05
Epoch: 3 cost time: 0.7926831245422363
Epoch: 3, Steps: 67 | Train Loss: 0.0001675 Vali Loss: 0.0001940 Test Loss: 0.0001816
Va

In [25]:
import pandas as pd
import os

df = pd.read_csv('dataset/LEAD_191_filtered.csv')

# Ensure temp directory exists
os.makedirs('temp', exist_ok=True)

# 1. Filter the dataframe to only include the target buildings
target_buildings = [55, 171, 190, 246, 312, 657, 903, 1074, 1143, 1241, 1242, 1246, 1260, 1266, 1267, 1275, 1278, 1279, 1297, 1319]
df_filtered = df[df['building_id'].isin(target_buildings)]

# 2. Loop through the unique building IDs found in the filtered data
for b_id in df_filtered['building_id'].unique():
    print(f"--- Processing building_id: {b_id} ---")
    
    # Filter data for this specific building
    df_b = df_filtered[df_filtered['building_id'] == b_id].copy()

    # Calculate anomaly ratio
    actual_ratio = df_b['anomaly'].mean() * 100 
    
    print(f"Calculated Anomaly Ratio for Building {b_id}: {actual_ratio:.2f}%")
    
    # Impute and Save
    df_b['meter_reading'] = df_b['meter_reading'].fillna(df_b['meter_reading'].median())
    df_b.to_csv('temp/train.csv', index=False)
    
    # Pass the ratio as an argument to the bash script
    !python -u run.py \
        --task_name anomaly_detection \
        --is_training 1 \
        --root_path ./temp \
        --data_path train.csv \
        --model_id LEAD \
        --model KANformerCorr \
        --data LEAD \
        --features S \
        --seq_len 168 \
        --pred_len 0 \
        --d_model 64 \
        --d_ff 64 \
        --e_layers 2 \
        --enc_in 1 \
        --c_out 1 \
        --anomaly_ratio {actual_ratio} \
        --batch_size 64 \
        --train_epochs 20 \
        --building_id {b_id} \
        --learning_rate 0.0001 \
        --patience 5

    print(f"Finished building_id: {b_id}\n")

--- Processing building_id: 55 ---
Calculated Anomaly Ratio for Building 55: 1.01%
Using GPU
Args in experiment:
Anomaly Detection Task
  Anomaly Ratio:      1.0132058287795993  

Use GPU: cuda:0
🚀 Lazy Loading: KANformerCorr ...
>>>>>>>start training : anomaly_detection_LEAD_KANformerCorr_LEAD_ftS_sl168_ll48_pl0_dm64_nh8_el2_dl1_df64_expand2_dc4_fc1_ebtimeF_dtTrue_test_0>>>>>>>>>>>>>>>>>>>>>>>>>>
train 4225
val 1297
test 2761
Epoch: 1 cost time: 1.1034879684448242
Epoch: 1, Steps: 67 | Train Loss: 0.0029746 Vali Loss: 0.0017077 Test Loss: 0.0024072
Validation loss decreased (inf --> 0.001708).  Saving model ...
Updating learning rate to 0.0001
Epoch: 2 cost time: 0.8121640682220459
Epoch: 2, Steps: 67 | Train Loss: 0.0007740 Vali Loss: 0.0007661 Test Loss: 0.0010490
Validation loss decreased (0.001708 --> 0.000766).  Saving model ...
Updating learning rate to 5e-05
Epoch: 3 cost time: 0.8497180938720703
Epoch: 3, Steps: 67 | Train Loss: 0.0005455 Vali Loss: 0.0005336 Test Loss: 0.0008

In [5]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"Is CUDA available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.11.0+cu130
Is CUDA available: True
GPU Count: 1
GPU Name: NVIDIA GeForce RTX 5090


In [13]:
# !python -c "import torch; print(torch.cuda.is_available())" True

In [5]:
df = pd.read_csv('dataset/LEAD_191_filtered.csv')
df.building_id.unique()

array([   1,   41,   55,   69,   79,   82,   91,  107,  108,  111,  112,
        117,  118,  119,  121,  136,  137,  139,  141,  144,  147,  148,
        149,  159,  171,  173,  174,  181,  183,  190,  235,  238,  240,
        246,  247,  248,  253,  254,  263,  270,  275,  276,  278,  290,
        293,  312,  318,  335,  345,  356,  423,  439,  492,  560,  623,
        657,  658,  666,  667,  673,  675,  677,  680,  683,  685,  687,
        697,  698,  701,  708,  710,  721,  722,  729,  730,  732,  742,
        801,  844,  848,  879,  880,  881,  882,  884,  886,  887,  889,
        890,  892,  893,  894,  895,  896,  903,  905,  909,  914,  919,
        922,  924,  925,  926,  928,  929,  931,  935,  936,  941,  942,
        945,  948,  950,  952,  961,  966,  967,  968,  969,  971,  973,
        974,  975,  977,  978,  981,  988,  990,  992,  994,  996, 1001,
       1007, 1068, 1073, 1074, 1106, 1120, 1128, 1137, 1141, 1143, 1172,
       1219, 1225, 1226, 1230, 1232, 1234, 1238, 12